# Planning a Two-Link 2D Arm to a Target Tip Position

This notebook walks through using `comb.planners.stepping.SteppingPlanner` to drive the 2D two-link arm so that its end-effector **tip** (the far end of `link_b`, not its body frame origin) reaches a specific world point. The pattern is:

1. Build the mode.
2. Express the goal as a *final constraint* — a `PointEquality2D` saying "the tip at offset `(L, 0)` in `link_b`'s frame coincides with the target world point". This is position-only (2 residuals), leaving the arm's orientation free.
3. Hand the mode + final constraint to the planner.
4. Animate the returned `Trajectory[ModeState]`, with the target point shown as a star marker.

In [ ]:
import numpy as np
from IPython.display import HTML
from matplotlib import animation, pyplot
from spatialmath import SE2

from comb.bodies import Body, BodyPoses, Rectangle
from comb.constraints import ConstraintParameters, PointEquality2D
from comb.examples.two_link_arm_2d import TwoLinkArm2D
from comb.planners.stepping import SteppingPlanner
from comb.rendering.matplotlib_2d import MatplotlibRenderer2D
from comb.rendering.overlays import PointMarker2D
from comb.mode import Mode
from comb.system import System

## 1. Build the arm and add an anchored world body

`TwoLinkArm2D` gives us a base + two links with revolute joints. To pin `link_b` at a target pose, we add an extra anchored *world* body that the final constraint will reference. The world body has empty geometry — it isn't drawn, just used as a reference frame.

In [2]:
arm = TwoLinkArm2D()

world = Body(
    name="world",
    pose=SE2(),
    visual_geometry=Rectangle(0.0, 0.0),
    collision_geometry=Rectangle(0.0, 0.0),
)

mode: Mode[SE2] = Mode(
    bodies=arm.mode.bodies + [world],
    constraints=list(arm.mode.constraints),
    configuration=arm.mode.configuration,
    body_poses=BodyPoses(
        {b: arm.mode.body_poses[b] for b in arm.mode.bodies} | {world: SE2()}
    ),
    anchored_bodies=arm.mode.anchored_bodies + [world],
)

## 2. Pick a target tip position

The "tip" is the far end of `link_b`, located at offset `(link_length, 0)` in `link_b`'s frame. We can pick any 2D world point inside the arm's reach (≤ `2 * link_length`).

In [ ]:
link_length = 1.0  # matches TwoLinkArm2D's default

target_x, target_y = 0.6, 1.4
print(f"Target tip position: ({target_x}, {target_y})")
print(
    f"Distance from base: {np.hypot(target_x, target_y):.3f} (must be ≤ {2 * link_length})"
)

## 3. Express the goal as a final constraint and plan

`PointEquality2D` says "this point in `body2`'s frame equals this point in `body1`'s frame". With `body1 = world` (anchored at origin) and `body2 = link_b`, the constraint `world.target == link_b.pose * (offset_x, offset_y)` becomes "the tip is at the target". Two residuals (x and y) — orientation stays free.

`interval` is the planner's main knob: it bounds how far any body's pose can move (in twist-norm distance) between adjacent solver checkpoints. Smaller values produce denser checkpoints and a path that hugs the constraint manifold more closely under linear interpolation.

In [ ]:
target_constraint = PointEquality2D(
    body1=world,
    body2=arm.link_b,
    fixed_parameters=ConstraintParameters(
        values=np.array([target_x, target_y, link_length, 0.0]),
        names=PointEquality2D.fixed_parameter_names(),
    ),
)

planner = SteppingPlanner(interval=0.1)
trajectory = planner.plan(System(mode=mode), [target_constraint], horizon=2.0)

goal_state = trajectory(trajectory.duration)
final_link_b = goal_state.body_poses[arm.link_b]
final_tip = final_link_b * SE2(link_length, 0.0, 0.0)
print(f"Trajectory duration: {trajectory.duration} s")
print(f"Final tip position: {tuple(final_tip.t)}")

## 4. Animate the trajectory

We sample the trajectory at fixed intervals via `Trajectory.enumerate(dt)` and render each frame onto a single matplotlib axes. The target tip position is drawn as a star marker via `PointMarker2D` so you can see exactly where the arm is reaching.

In [ ]:
fig, ax = pyplot.subplots(figsize=(5, 5))
renderer = MatplotlibRenderer2D(ax=ax)

samples = list(trajectory.enumerate(0.05))
target_marker = PointMarker2D(
    x=target_x, y=target_y, marker="*", color="tab:orange", size=300.0
)


def draw(frame_idx: int):
    _, state = samples[frame_idx]
    for body in arm.mode.bodies:
        arm.mode.body_poses[body] = state.body_poses[body]
    for c in arm.mode.configuration:
        arm.mode.configuration[c] = state.configuration[c]
    renderer.render(arm.mode, overlays=[target_marker])
    return []


anim = animation.FuncAnimation(fig, draw, frames=len(samples), interval=50)
pyplot.close(fig)
HTML(anim.to_jshtml())

## What to try next

- Tighten `interval` (e.g. `0.02`) and replay — the trajectory should look smoother and the path stays closer to the constraint manifold.
- Pick a target outside the arm's reach (e.g. `(2.5, 0)`). `SteppingPlanner.plan` will surface this as a `RuntimeError` from `find_satisfying_state`.
- Pin both the tip *and* the link orientation: combine a `PointEquality2D` with a `FixedJoint2D` on the same body, or replace `PointEquality2D` with a single `FixedJoint2D` if you want full SE(2) lock-in (over-constrained for a 2-DoF arm — only specific poses are reachable).
- Chain plans: `mode.apply(traj(traj.duration))` then call `plan` again with a new target. `concatenate(...)` joins them into one `Trajectory`.